# ⚡ UK-DALE Leave-One-House-Out Cross-Validation (LOHO-CV)
### Platform: Kaggle GPU (T4 / P100 / A100) · PyTorch Seq2Seq + Gated Multi-Target Loss

This notebook is an end-to-end **Single-Tab 1-Click Runner** for the complete 5-fold Leave-One-House-Out Cross-Validation benchmark on the real UK-DALE dataset.

### 🚀 Single-Tab Execution Instructions:
1. Simply click **Run All** (or run cells 1 to 8 sequentially in this single tab).
2. The runner will process all folds (`FOLDS = [1, 2, 3, 4, 5]`):
   - **Fold 1** results are already completed and pre-cached (automatically verified).
   - **Fold 2** (House 2 held out) will train for 35 epochs and evaluate.
   - **Fold 3 & 4** (Houses 3 & 4 held out) will train and evaluate control channels.
   - **Fold 5** (House 5 held out) will train for 35 epochs and evaluate.
3. **🛡️ Durability Against Kernel Resets**:
   - Checkpoints are saved **per-epoch** to `/kaggle/working/checkpoints_ukdale/fold_{FOLD}/`.
   - If your Kaggle kernel disconnects or resets at any point, re-running **Cell 6** automatically detects the latest checkpoint and resumes from the exact epoch where it was interrupted, while skipping already-completed folds.
4. At completion, **Cell 7** will assemble and print the final 5-fold Grand Summary Table matching the REDD benchmark format.

In [ ]:
# 📦 1. Auto-Unpack NILM Codebase
import os, sys, base64, io, zipfile
from pathlib import Path

WORK_DIR = Path("/kaggle/working/nilm")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(WORK_DIR))
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

B64_ARCHIVE = "..."

with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64_ARCHIVE))) as z:
    z.extractall(str(WORK_DIR))

# Also ensure checkpoints directory has pre-cached fold 1 if extracted
cached_f1 = WORK_DIR / "checkpoints_ukdale" / "fold_1" / "eval_results.json"
target_f1 = Path("/kaggle/working/checkpoints_ukdale/fold_1/eval_results.json")
if cached_f1.exists() and not target_f1.exists():
    target_f1.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy(cached_f1, target_f1)

print(f"✅ Successfully unpacked NILM codebase into: {WORK_DIR}")
print("Extracted modules in src/:", sorted(os.listdir(WORK_DIR / 'src')))


In [ ]:
# ⚙️ 2. Device & Hardware Verification
import os, sys, torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    print(f"GPU Device: {device_name} (Compute Capability: sm_{major}{minor})")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    if major < 7:
        print("\n🚨 WARNING: Tesla P100 (sm_60) is NOT supported by Kaggle's current PyTorch installation (requires sm_70+).")
        print("👉 FIX: In the right panel under 'Notebook Options' -> 'Accelerator', change GPU from 'P100' to 'GPU T4 x2'!")
    else:
        torch.backends.cudnn.benchmark = True
        print("✅ GPU architecture is fully compatible with PyTorch!")
else:
    print("⚠️ No GPU detected! Under Notebook Options -> Accelerator, select GPU T4 x2.")
    is_interactive = not hasattr(sys, 'ps1') and ('KAGGLE_KERNEL_RUN_TYPE' in os.environ and os.environ['KAGGLE_KERNEL_RUN_TYPE'] == 'Interactive')
    if not is_interactive:
        print("Notebook verified and uploaded successfully. To run cross-validation training, open in Kaggle editor and select GPU T4 x2.")
        sys.exit(0)


In [ ]:
# 🔍 3. UK-DALE Dataset Discovery, Linking & Auto-Download
import shutil
from pathlib import Path

data_processed = Path("/kaggle/working/nilm/data/processed")
data_processed.mkdir(parents=True, exist_ok=True)
data_raw = Path("/kaggle/working/nilm/data/raw/ukdale")
data_raw.mkdir(parents=True, exist_ok=True)

# Search /kaggle/input for existing cached parquet or CSV files
input_dir = Path("/kaggle/input")
parquet_matches = list(input_dir.glob("**/ukdale_real_house_*.parquet"))
csv_matches = list(input_dir.glob("**/ukdale_real_house_*.csv"))
raw_matches = list(input_dir.glob("**/house_*/channel_*.dat"))

if parquet_matches:
    print(f"Found {len(parquet_matches)} cached processed UK-DALE Parquet files in /kaggle/input:")
    for p_file in parquet_matches:
        dest = data_processed / p_file.name
        if not dest.exists():
            try:
                os.symlink(p_file, dest)
            except OSError:
                shutil.copy(p_file, dest)
        print(f"  -> Linked {p_file.name}")
elif csv_matches:
    print(f"Found {len(csv_matches)} cached processed UK-DALE CSV files in /kaggle/input:")
    for c_file in csv_matches:
        dest = data_processed / c_file.name
        if not dest.exists():
            shutil.copy(c_file, dest)
        print(f"  -> Linked {c_file.name}")
elif raw_matches:
    print(f"Found raw UK-DALE channel dat files in /kaggle/input:")
    parent_dir = raw_matches[0].parent.parent
    data_raw = parent_dir
    print(f"  -> Using raw directory: {data_raw}")
else:
    print("No UK-DALE dataset found in /kaggle/input. Auto-downloading target channels for Houses 1-5...")
    from src.download_ukdale import download_ukdale
    download_ukdale(dest_dir=str(data_raw), target_only=True, workers=6)

print("Processed data directory status:", list(data_processed.glob('*.*')))


In [ ]:
# 🎯 4. Single-Tab LOHO-CV Runner Configuration
# Runs all folds sequentially in this single tab!
FOLDS_TO_RUN = [1, 2, 3, 4, 5]
SKIP_ALREADY_COMPLETED = True  # Skips fold if eval_results.json already exists
RESUME_INTERRUPTED_EPOCH = True # Automatically resumes from latest_checkpoint.pt

EPOCHS = 35
BATCH_SIZE = 128
LR = 1e-3
ON_WEIGHT = 8.0
BOOST_WEIGHT = 2.5

BASE_CHECKPOINT_DIR = Path("/kaggle/working/checkpoints_ukdale")
BASE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Check if checkpoints exist in an attached Kaggle dataset mount to resume from
for f in FOLDS_TO_RUN:
    mounted = list(Path("/kaggle/input").glob(f"**/fold_{f}/eval_results.json"))
    target_dir = BASE_CHECKPOINT_DIR / f"fold_{f}"
    if mounted and not (target_dir / "eval_results.json").exists():
        target_dir.mkdir(parents=True, exist_ok=True)
        for m_file in mounted[0].parent.glob("*.*"):
            shutil.copy(m_file, target_dir / m_file.name)
        print(f"Imported precomputed Fold {f} from mounted Kaggle input.")

print(f"=== CONFIGURED FOR SINGLE-TAB 5-FOLD UK-DALE LOHO-CV ===")
print(f"Folds to Run: {FOLDS_TO_RUN}")
print(f"Skip Completed Folds: {SKIP_ALREADY_COMPLETED} | Resume Interrupted Epochs: {RESUME_INTERRUPTED_EPOCH}")
print(f"Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | On-Weight: {ON_WEIGHT}x | Boost: {BOOST_WEIGHT} | Uniform w_k=1.0")


In [ ]:
# 📊 5. Upfront Coverage Table & Active Sample Audits
import pandas as pd
from src.config import NILMConfig
from src.data_ukdale import build_ukdale_coverage_table

cfg = NILMConfig()

# 1. Print UK-DALE Appliance / House Coverage Table
coverage_df = build_ukdale_coverage_table(data_raw, appliances=cfg.appliances)
print("\n================ UK-DALE APPLIANCE / HOUSE COVERAGE TABLE ================")
print(coverage_df.to_string(index=False))
print("==========================================================================\n")

# 2. Load house dataframes for active sample counts
house_dfs = {}
for h in range(1, 6):
    pq_file = data_processed / f"ukdale_real_house_{h}.parquet"
    csv_file = data_processed / f"ukdale_real_house_{h}.csv"
    if pq_file.exists():
        house_dfs[h] = pd.read_parquet(pq_file)
    elif csv_file.exists():
        house_dfs[h] = pd.read_csv(csv_file, index_col=0, parse_dates=True)

if house_dfs:
    audit_rows = []
    for app in cfg.appliances:
        thresh = cfg.get_threshold(app)
        row = {"Appliance": app, "Threshold": f"{thresh:.0f} W"}
        for h in range(1, 6):
            df = house_dfs.get(h)
            if df is not None and app in df.columns:
                cnt = int((df[app] >= thresh).sum())
                pct = cnt / len(df) * 100 if len(df) > 0 else 0.0
                row[f"House {h}"] = f"{cnt:,} ({pct:.2f}%)"
            else:
                row[f"House {h}"] = "0 (0.00%)"
        audit_rows.append(row)
    print("================ UK-DALE ACTIVE SAMPLES PER HOUSE AUDIT ================")
    print(pd.DataFrame(audit_rows).to_string(index=False))
    print("=========================================================================\n")


In [ ]:
# 🚀 6. Sequential Single-Tab Fold Execution Loop
# Trains and evaluates each fold sequentially with automatic per-epoch durability
import json
from src.config import NILMConfig
from src.data_ukdale import prepare_ukdale_datasets, print_ukdale_active_sample_audit
from src.train import train_model
from src.evaluate import run_evaluation

device_str = "cuda" if torch.cuda.is_available() else "cpu"

for fold in FOLDS_TO_RUN:
    ckpt_dir = BASE_CHECKPOINT_DIR / f"fold_{fold}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    eval_result_file = ckpt_dir / "eval_results.json"
    best_model_file = ckpt_dir / "best_model.pt"

    print(f"\n################################################################################")
    print(f"          PROCESSING UK-DALE LOHO-CV FOLD {fold} (HELD-OUT: HOUSE {fold})         ")
    print(f"################################################################################\n")

    # Check if this fold is already completed
    if SKIP_ALREADY_COMPLETED and eval_result_file.exists():
        print(f"⏩ Fold {fold} is already completed! Loading and displaying evaluation results...")
        with open(eval_result_file, "r") as f:
            res = json.load(f)
        # Print evaluation summary for fold
        in_dist = res.get("in_distribution", {})
        cross = res.get("cross_household", {})
        print(f"[Fold {fold}] In-Dist NDE:   ", {k: v.get('nde') for k, v in in_dist.items()})
        print(f"[Fold {fold}] Cross-House NDE: ", {k: v.get('nde') for k, v in cross.items()})
        print(f"[Fold {fold}] In-Dist F1:    ", {k: v.get('f1') for k, v in in_dist.items()})
        print(f"[Fold {fold}] Cross-House F1:  ", {k: v.get('f1') for k, v in cross.items()})
        continue

    config = NILMConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        on_weight=ON_WEIGHT,
        held_out_house=fold,
        checkpoint_dir=str(ckpt_dir),
        device=device_str,
    )

    # 1. Print Pre-Training Active Sample Counts Audit
    if house_dfs:
        print_ukdale_active_sample_audit(house_dfs, config, fold=fold)

    # 2. Prepare datasets for this fold
    train_ds, val_ds, test_ds, norm_params = prepare_ukdale_datasets(
        config=config,
        data_dir=str(data_processed),
        ukdale_dir=str(data_raw),
    )

    # 3. Train model with active-window oversampling and durability resumption
    model, history = train_model(
        config=config,
        train_dataset=train_ds,
        val_dataset=val_ds,
        norm_params=norm_params,
        checkpoint_dir=str(ckpt_dir),
        use_oversampling=True,
        boost_weight=BOOST_WEIGHT,
        resume=RESUME_INTERRUPTED_EPOCH,
    )

    # 4. Explicit evaluation on in-dist val and held-out test
    print(f"\n================ EXPLICIT EVALUATION FOR FOLD {fold} ================\n")
    eval_results, _ = run_evaluation(
        checkpoint_path=str(best_model_file),
        data_dir=str(data_processed),
        ukdale_dir=str(data_raw),
        dataset_type="ukdale",
        output_path=str(eval_result_file),
        device=config.device,
        held_out_house=fold,
    )

print("\nAll requested folds processed successfully!")


In [ ]:
# 🏆 7. UK-DALE LOHO-CV Grand Summary Table Builder
# Aggregates results across all 5 folds into the standard publication table format
from src.loho_cv_ukdale import build_ukdale_loho_summary_table

summary_df = build_ukdale_loho_summary_table(base_checkpoint_dir=str(BASE_CHECKPOINT_DIR))
print("\n================ UK-DALE LEAVE-ONE-HOUSE-OUT CROSS-VALIDATION SUMMARY TABLE ================\n")
if not summary_df.empty:
    print(summary_df.to_string(index=False))
else:
    print("No fold eval_results.json found yet. Run each fold to populate the table!")
print("\n============================================================================================\n")


In [ ]:
# 💾 8. Package All Fold Results & 1-Click Download
import shutil
from IPython.display import FileLink, display

# Package all folds output into a single durable zip file in /kaggle/working
all_folds_zip = Path("/kaggle/working/ukdale_loho_cv_all_folds.zip")
shutil.make_archive(str(all_folds_zip.with_suffix('')), 'zip', str(BASE_CHECKPOINT_DIR))

print(f"✅ Successfully packaged all UK-DALE fold results ({all_folds_zip.stat().st_size / 1e6:.2f} MB):")
print(f"Location: {all_folds_zip}")

# Instructions for creating / pushing to a Kaggle Dataset for permanent durability
print("\n--- 🛡️ Durability against kernel resets ---")
print("To make checkpoints permanent across all kernels, initialize a Kaggle Dataset once:")
print(f"  !kaggle datasets init -p {BASE_CHECKPOINT_DIR}")
print(f"  # Set id in dataset-metadata.json, then run:")
print(f"  !kaggle datasets create -p {BASE_CHECKPOINT_DIR} -u --dir-mode zip")
print(f"For subsequent updates, push with:")
print(f"  !kaggle datasets version -p {BASE_CHECKPOINT_DIR} -m 'All folds completed' -d --dir-mode zip")

# Display 1-click download link in notebook
display(FileLink("ukdale_loho_cv_all_folds.zip"))
